# 第15周 · 练习：重参数化、KL 项与插值

VAE 的三个零件各写一遍，再把它们装进 pytorch/examples 的 VAE 骨架里训练：

| 题 | 内容 | 自查 |
| --- | --- | --- |
| E1 | 重参数化 `reparameterize(mu, logvar)` | `ck.reparameterize` |
| E2 | KL 项 `vae_kl(mu, logvar)`，并与第 14 周的通用高斯 KL 对账 | `ck.vae_kl` |
| E3 | 潜空间插值 `interpolate(z1, z2, n)` | `ck.interpolate` |
| E4 | 用你的三个函数组装并训练 MNIST VAE | `ck.vae_test_loss` |
| E5 | 把 KL 项拿掉（β = 0）会怎样？ | 观察 + 回答 |

全部在 CPU 上几秒钟就能跑完。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from zero2researcher import data, experiments
from zero2researcher.checks import week15 as ck
from zero2researcher.checks import report

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## E1 · 重参数化

编码器输出 `mu` 和 `logvar`（$= \log\sigma^2$），形状都是 `(batch, 潜维度)`。写 `reparameterize(mu, logvar)`，返回同形状的 $z = \mu + \sigma \cdot \varepsilon$，其中 $\sigma = \exp(\frac{1}{2}\,\text{logvar})$、$\varepsilon \sim \mathcal{N}(0, 1)$ 每次调用重新抽。

自查会检查：样本均值≈`mu`、样本标准差≈$\sigma$；$z$ 能对 `mu`、`logvar` 求导；两次调用的结果不同。

In [ ]:
def reparameterize(mu, logvar):
    # 在这里写你的代码
    raise NotImplementedError


ck.reparameterize(reparameterize)

**追问**（一句话）：如果把 `std = torch.exp(0.5 * logvar)` 错写成 `torch.exp(logvar)`，采样出来的 $z$ 的标准差会变成多少？自查是怎么发现的？

（在这里回答）

## E2 · KL 项

$$\mathrm{KL}\big(\mathcal{N}(\mu, \sigma^2)\,\|\,\mathcal{N}(0, I)\big) = -\frac{1}{2}\sum_{\text{batch}}\sum_{j}\left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

写 `vae_kl(mu, logvar)`，返回一个标量：对 batch 和潜维度**全部求和**（与 BCE 的 `reduction="sum"` 同一口径，pytorch/examples 的写法）。然后和第 14 周的一维通用公式对账：把每个 $(\mu_j, \sigma_j)$ 代进 `gaussian_kl(mu, sigma, 0, 1)` 逐个相加，两者应相等。

In [ ]:
def vae_kl(mu, logvar):
    # 在这里写你的代码
    raise NotImplementedError


def gaussian_kl(mu1, sigma1, mu2, sigma2):
    # 第 14 周练习 E4 的参考写法
    return math.log(sigma2 / sigma1) + (sigma1 ** 2 + (mu1 - mu2) ** 2) / (2 * sigma2 ** 2) - 0.5


torch.manual_seed(0)
mu, logvar = torch.randn(3, 4), torch.randn(3, 4)
# 在这里写你的代码
raise NotImplementedError
print(f"vae_kl = {vae_kl(mu, logvar).item():.5f} | 逐元素用通用公式相加 = {by_hand:.5f}")
ck.vae_kl(vae_kl)

## E3 · 潜空间插值

写 `interpolate(z1, z2, n)`：`z1`、`z2` 是形状 `(d,)` 的两个潜向量，返回 `(n, d)` 的张量——第 0 行是 `z1`、最后一行是 `z2`，中间等距线性插值：$z(t) = (1 - t)\,z_1 + t\,z_2$，$t$ 从 0 到 1 取 `n` 个值。

提示：`torch.linspace(0, 1, n)` 的形状是 `(n,)`，加一个维度变成 `(n, 1)` 就能和 `(d,)` 广播。

In [ ]:
def interpolate(z1, z2, n):
    # 在这里写你的代码
    raise NotImplementedError


print(interpolate(torch.tensor([0.0, 0.0]), torch.tensor([1.0, -2.0]), 5))
ck.interpolate(interpolate)

## E4 · 组装并训练 MNIST VAE

下面是 pytorch/examples/vae（BSD-3-Clause）的 VAE 骨架，`forward` 里调用的是**你的** `reparameterize`。补全 `train_epoch` 里的损失：重建项 `F.binary_cross_entropy(recon, xb, reduction="sum")` + 你的 `vae_kl`。在 MNIST 前 1 万张上训练 5 个 epoch，报告测试集上**每张图**的损失。

In [ ]:
def load_dataset(cls, train):
    for attempt in (1, 2):
        try:
            return cls(root=data.download_root(), train=train, download=True)
        except Exception as e:
            if attempt == 1:
                print("下载失败，重试一次：", e)
            else:
                raise


mnist_train, mnist_test = load_dataset(datasets.MNIST, True), load_dataset(datasets.MNIST, False)
X_train = mnist_train.data[:10000].float().div(255).reshape(-1, 784)
X_test = mnist_test.data.float().div(255).reshape(-1, 784)
y_test = mnist_test.targets


class VAE(nn.Module):
    def __init__(self, z_dim=20):
        super().__init__()
        self.fc1, self.fc21, self.fc22 = nn.Linear(784, 400), nn.Linear(400, z_dim), nn.Linear(400, z_dim)
        self.fc3, self.fc4 = nn.Linear(z_dim, 400), nn.Linear(400, 784)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc21(h), self.fc22(h)

    def decode(self, z):
        return torch.sigmoid(self.fc4(F.relu(self.fc3(z))))

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(reparameterize(mu, logvar)), mu, logvar


def train_epoch(model, loader, optimizer, beta=1.0):
    model.train()
    for (xb,) in loader:
        xb = xb.to(device)
        recon, mu, logvar = model(xb)
        # 在这里写你的代码
        raise NotImplementedError
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


def test_loss(model, X):
    # 每张图的 (BCE, KL)
    model.eval()
    with torch.no_grad():
        recon, mu, logvar = model(X.to(device))
        return (F.binary_cross_entropy(recon, X.to(device), reduction="sum").item() / len(X),
                vae_kl(mu, logvar).item() / len(X))


def fit(beta=1.0, epochs=5, seed=0):
    experiments.set_seed(seed)
    model = VAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(X_train), batch_size=128, shuffle=True)
    for _ in range(epochs):
        train_epoch(model, loader, optimizer, beta)
    return model


vae = fit()
bce, kl = test_loss(vae, X_test)
print(f"测试集每张图：BCE {bce:.1f} + KL {kl:.1f} = {bce + kl:.1f}")
ck.vae_test_loss(bce + kl)

用你的 `interpolate` 在一个 "4" 和一个 "9" 之间插值 8 步（用编码器的 `mu` 作为端点），把解码结果画成一行：

In [ ]:
i4, i9 = (y_test == 4).nonzero()[0].item(), (y_test == 9).nonzero()[0].item()
with torch.no_grad():
    mu_pair = vae.encode(X_test[[i4, i9]].to(device))[0]
    # 在这里写你的代码
    raise NotImplementedError
fig, axes = plt.subplots(1, 8, figsize=(8, 1.3))
for ax, img in zip(axes, frames):
    ax.imshow(img.reshape(28, 28), cmap="gray_r")
    ax.axis("off")
fig.suptitle("Interpolating from a 4 to a 9 in latent space", fontsize=9)
plt.show()

## E5 · 把 KL 项拿掉会怎样？

`fit(beta=0.0)` 就是去掉 KL 项、只剩重建：VAE 退化成一个"编码器会加点噪声"的自编码器。训练它，和 E4 的 β = 1 模型比较两件事：

1. 测试集上每张图的 KL（虽然训练时不罚它，照样可以量）；
2. 从先验 $\mathcal{N}(0, I)$ 采样 16 个 $z$ 解码出来的图。

In [ ]:
vae_b0 = fit(beta=0.0)
bce0, kl0 = test_loss(vae_b0, X_test)
print(f"β = 0：BCE {bce0:.1f} | KL {kl0:.1f}      β = 1：BCE {bce:.1f} | KL {kl:.1f}")
report("去掉 KL 项后，编码偏离先验", kl0 > 2 * kl, f"β = 0 的 KL（{kl0:.1f}）应远大于 β = 1（{kl:.1f}）",
       "fit(beta=0.0) 是否真的把 beta 传进了 train_epoch？")

torch.manual_seed(0)
z = torch.randn(16, 20, device=device)
fig, axes = plt.subplots(2, 16, figsize=(12, 1.8))
with torch.no_grad():
    for row, (model, name) in enumerate(((vae, "beta = 1"), (vae_b0, "beta = 0"))):
        for ax, img in zip(axes[row], model.decode(z).cpu()):
            ax.imshow(img.reshape(28, 28), cmap="gray_r")
            ax.axis("off")
        axes[row, 0].set_title(name, fontsize=8, loc="left")
fig.suptitle("Same prior samples: without the KL term the decoder has never seen these z", fontsize=9)
plt.show()

**回答**（两三句话）：β = 0 时重建变好了还是变差了？为什么从先验采样的图反而更差？用"潜空间里的空地"解释。由此预测：作业里 β 从 0.1 增大到 4，重建误差、KL、以及潜空间的结构会怎样变化？

（在这里回答）